# CEDAR Query API - Example Search Scenarios and Queries

The Cancer Epitope Database and Analysis Resource (CEDAR) API (Application Programming Interface) enables users to perform queries and retrieve CEDAR data programmatically within their preferred environment. This Python Notebook Series illustrates basic usage examples of the CEDAR API using Python.

The CEDAR API is built on a PostgREST platform, providing transparent access to the tables on the backend. For more information on the expressive syntax of PostgREST, refer to this [document](https://postgrest.org/en/stable/api.html#). The CEDAR data can be queried through search and export endpoints described in the [CEDAR API documentation](https://help.iedb.org/hc/en-us/articles/4402872882189-Immune-Epitope-Database-Query-API-IQ-API) and the interactive [Swagger documentation](https://cedar-api.iedb.org/docs/swagger/).

These example search scenarios and queries are derived from this [Book Chapter](https://doi.org/10.1007/978-1-0716-4566-6_3), which describes how to perform different queries to the [CEDAR homepage](https://cedar.iedb.org). This notebook reproduces the same queries using the CEDAR API.

## Research scenario II

The user conducted a study about recurrent mutations in cancer and identified putative neoantigens using prediction tools. The neoepitope “SYLDSGIHF” from Catenin beta 1 (CTNNB1) was found to be recurring in many patients and was predicted to be potentially immunogenic. The user wants to investigate the experimental evidence for immunogenicity reported in the literature.

### Approach

To retrieve this data, we will execute the following steps:

1. **Search the neoepitope** - Identify assay IDs in which the neoepitope has been tested
2. **Map associated MHC ligand assays** - Retrieve comprehensive assay information for MHC ligand experiments
3. **Map associated T cell assays** -  Retrieve comprehensive assay information for T cell experiments

Each step requires a request to a different interconnected endpoint. For more information, refer to the [PostgREST resource embedding documentation](https://docs.postgrest.org/en/v12/references/api/resource_embedding.html#resource-embedding).

### Setup

First, import the required modules and define a function to make CEDAR API requests. This function takes the endpoint, the search parameters, and the base URI (Unified Resource Identifier) as inputs. Based on these, it constructs the URL (Unified Resource Locator) to interact with the data and resources specified. Then, it performs the API request. Since the API has a maximum limit of 10,000 rows per request, the function iterates through results, retrieving data in chunks. Finally, it parses the response and returns a pandas DataFrame. For more information, refer to [pandas documentation](https://pandas.pydata.org/).

In [1]:
import os
import requests
import pandas as pd
import io
import time

pd.set_option('display.max_columns', 100)

def API_query(endpoint, query_params, base_uri='https://cedar-api.iedb.org/'):
    """
    Execute a query against the CEDAR API with pagination support.

    Parameters:
    -----------
    endpoint : str
        The API endpoint to query
    query_params : dict
        Dictionary of query parameters
    base_uri : str
        Base URI for the CEDAR API

    Returns:
    --------
    pd.DataFrame
        Combined results from all paginated requests
    """

    url = os.path.join(base_uri, endpoint)
    df = pd.DataFrame()

    # set the offset to 0
    query_params['offset'] = 0

    # loop through the pages of results
    # API only allows pulling 10,000 entries at a time
    while(True):
        print('Fetching offset: %i' % query_params['offset'])
        r = requests.get(
            url,
            params=query_params,
            headers={'accept': 'text/csv', 'Prefer': 'count=exact'}
        )

        try:
            # Parse CSV response and append to existing DataFrame
            df = pd.concat([df, pd.read_csv(io.StringIO(r.content.decode('utf-8')))])
            query_params['offset'] += 10000
        except pd.errors.EmptyDataError:
            # No more data available
            break

        # Rate limiting: pause between requests to not overload the server
        time.sleep(1)

    return df


### Step 1: Search Epitope

In this example, we are searching for assays associated with a specific **neoepitope**.

The CEDAR API provides two types of endpoints, the search and export. The search endpoints contain fields with information that facilitate to programatically identify and/or filter the data of interest, while the export endpoints organize the data in a user-friendly format, matching the structure and naming conventions of the custom exports on the CEDAR website.

To perform our query, we will first use the `epitope_search` endpoint. Let's examine its structure and available fields.


In [2]:
cedar_url='https://cedar-api.iedb.org/'
endpoint='epitope_search'

table = pd.json_normalize(requests.get(os.path.join(cedar_url, endpoint + '?limit=3')).json())
display(table)

,structure_id,structure_iri,structure_descriptions,curated_source_antigens,structure_type,linear_sequence,e_modification,linear_sequence_length,cedar_assay_ids,cedar_assay_iris,reference_ids,reference_iris,submission_ids,submission_iris,pdb_ids,chebi_ids,qualitative_measures,mhc_allele_evidences,antibody_isotypes,direct_ex_vivo_bool,receptor_ids,receptor_group_ids,tcr_receptor_group_ids,bcr_receptor_group_ids,receptor_group_iris,tcr_receptor_group_iris,bcr_receptor_group_iris,receptor_types,receptor_names,receptor_chain1_types,receptor_chain2_types,receptor_chain1_full_seqs,receptor_chain2_full_seqs,receptor_chain1_cdr1_seqs,receptor_chain2_cdr1_seqs,receptor_chain1_cdr2_seqs,receptor_chain2_cdr2_seqs,receptor_chain1_cdr3_seqs,receptor_chain2_cdr3_seqs,host_organism_iri_search,host_organism_iris,host_organism_names,source_organism_iri_search,source_organism_iris,source_organism_names,mhc_allele_iri_search,mhc_allele_iris,mhc_allele_names,parent_source_antigen_iri_search,parent_source_antigen_iris,parent_source_antigen_names,parent_source_antigen_source_org_iris,parent_source_antigen_source_org_names,disease_iri_search,disease_iris,disease_names,assay_iri_search,assay_iris,assay_names,epitope_structures_defined,non_peptidic_molecule_iri_search,non_peptidic_molecule_iris,non_peptidic_molecule_names,r_object_source_molecule_iri_search,r_object_source_molecule_iris,r_object_source_molecule_names,r_object_source_organism_iri_search,r_object_source_organism_iris,r_object_source_organism_names,mhc_classes,mhc_allele_resolutions,e_related_object_types,tcell_ids,tcell_iris,bcell_ids,bcell_iris,elution_ids,elution_iris,journal_names,reference_types,pubmed_ids,reference_titles,reference_authors,reference_dates,neoantigen_bool,viral_antigen_bool,germline_antigen_bool,other_antigen_bool,disease_stages,naturally_occuring_disease_bool,animal_model_of_cancer_bool,all_vaccination_bool,prophylactic_vaccination_bool,therapeutic_vaccination_bool,mutation
0,1635192,CEDAR_EPITOPE:1635192,[PEPTKSAPAPKKGSKKAVTKAQKKDGKKR],"[{'accession': 'P10854.2', 'name': 'Histone H2...",Linear peptide,PEPTKSAPAPKKGSKKAVTKAQKKDGKKR,None,29,[16582809],[CEDAR_ASSAY:16582809],[1039608],[CEDAR_REFERENCE:1039608],None,None,None,None,[Positive],[Allele/Locus-specific Antibody],None,0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"[NCBITaxon:10090, NCBITaxon:314147, NCBITaxon:...",[taxon:10000067],[Mus musculus C57BL/6],"[NCBITaxon:1, NCBITaxon:10090, NCBITaxon:2759,...",[NCBITaxon:10090],[Mus musculus (mouse)],"[GO:0032991, GO:0042611, MRO:0000010, MRO:0000...",[MRO:0000972],[H2-IAb],"[PR:000000001, taxon_protein:10090, taxon_prot...",[UNIPROT:P10854],[Histone H2B type 1-M (UniProt:P10854)],[NCBITaxon:10090],[Mus musculus (mouse)],None,None,None,"[OBI:0001478, OBI:0001489, OBI:0002075, OBI:11...",[OBI:0001489],[ligand presentation|cellular MHC/mass spectro...,[Epitope containing region/antigenic site],None,None,None,None,None,None,None,None,None,[II],[2 chain],None,None,None,None,None,[16582809],[CEDAR_ASSAY:16582809],[Immunity],[Literature],[33725478],[Pleiotropic consequences of metabolic stress ...,[Cristina C Clement; Padma P Nanaware; Takahir...,[2021],0,0,0,1,None,0,0,0,0,0,None
1,1635196,CEDAR_EPITOPE:1635196,[PFRVTEQESKPVQ],"[{'accession': 'P01012.2', 'name': 'Ovalbumin'...",Linear peptide,PFRVTEQESKPVQ,None,13,[16582198],[CEDAR_ASSAY:16582198],[1039608],[CEDAR_REFERENCE:1039608],None,None,None,None,[Positive],[Allele/Locus-specific Antibody],None,0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,"[NCBITaxon:10090, NCBITaxon:314147, NCBITaxon:...",[taxon:10000067],[Mus musculus C57BL/6],"[NCBITaxon:1, NCBITaxon:2759, NCBITaxon:33208,...",[NCBITaxon:9031],[Gallus gallus (chicken)],"[GO:0032991, GO:0042611, MRO:0000010, MRO:0000...",[MRO:0000972],[H2-IAb],"[PR:000000001, taxon_protein:2759, taxon_prote...",[UNIPROT:P01012],[Ovalbumin (UniProt:P01012)],[NCBITaxon:9031],[Ga

Next, we define the search parameters, which consist of three components:

- The **query filters** that we want to impose on the data. Here, we will filter linear peptides matching the amino acid sequence of interest. To do this, we utilize operators, such as `eq`, that allow us to perform flexible queries and refine the search. For more information on the operators' syntax, refer to this [documentation](https://docs.postgrest.org/en/v12/references/api/tables_views.html#operators).
- The **pagination parameters** required to retrieve the results in chunks. The order parameter is a field used to sort and paginate the data so that each retrieved chunk contains different consecutive rows. The offset parameter indicates the index of the first element of each data chunk.
- The final **column selection**. In this case, we will keep the assay IDs of the `iedb_assay_ids` field. These will be used for subsequent queries to other endpoints.

In [3]:
search_params={
    # Query filters
    'structure_type':'eq.Linear peptide',
    'linear_sequence':'eq.SYLDSGIHF',

    # Pagination
    'order': 'structure_id',
    'offset': 0,

    # Column selection
    'select':'cedar_assay_ids',
}

# Execute epitope search
epitope_df = API_query(endpoint, search_params)

print(f"Retrieved {len(epitope_df)} epitope records")
epitope_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 1 epitope records


,cedar_assay_ids
0,"{3000175,7689417,21249142,21249144,21249146,21..."


The search identified 1 record in the ``epitope_search`` endpoint corresponding to the neoepitope of interest, and retrieved all the assay IDs where this epitope has been tested across different assay types.

### Step 2: Map associated MHC elution assays

With the assay IDs of interest, we can map the information of associated MHC ligand assays by querying the `mhc_export` endpoint. Let's examine its structure and available fields.

In [4]:
endpoint='mhc_export'

table = pd.json_normalize(requests.get(os.path.join(cedar_url, endpoint + '?limit=3')).json())
display(table)

,assay_id,assay_id__cedar_iri,reference__cedar_iri,reference__type,reference__pmid,reference__submission_id,reference__authors,reference__journal,reference__date,reference__title,epitope__epitope_iri,epitope__object_type,epitope__name,epitope__reference_name,epitope__modified_residues,epitope__modifications,epitope__starting_position,epitope__ending_position,epitope__iri,epitope__synonyms,epitope__source_molecule,epitope__source_molecule_iri,epitope__molecule_parent,epitope__molecule_parent_iri,epitope__source_organism,epitope__source_organism_iri,epitope__species,epitope__species_iri,epitope__mutation,epitope__comments,related_object__epitope_relation,related_object__object_type,related_object__name,related_object__starting_position,related_object__ending_position,related_object__iri,related_object__synonyms,related_object__source_molecule,related_object__source_molecule_iri,related_object__molecule_parent,related_object__molecule_parent_iri,related_object__source_organism,related_object__source_organism_iri,related_object__species,related_object__species_iri,host__name,host__iri,host__geolocation,host__geolocation_iri,host__sex,...,in_vivo_antigen__source_molecule_iri,in_vivo_antigen__molecule_parent,in_vivo_antigen__molecule_parent_iri,in_vivo_antigen__source_organism,in_vivo_antigen__source_organism_iri,in_vivo_antigen__species,in_vivo_antigen__species_iri,in_vivo_antigen__adjuvants,in_vivo_antigen__route,in_vivo_antigen__dose_schedule,in_vitro_process__process_type,in_vitro_process__epitope_relation,in_vitro_process__object_type,in_vitro_process__name,in_vitro_process__reference_name,in_vitro_process__starting_position,in_vitro_process__ending_position,in_vitro_process__iri,in_vitro_process__source_molecule,in_vitro_process__source_molecule_iri,in_vitro_process__molecule_parent,in_vitro_process__molecule_parent_iri,in_vitro_process__source_organism,in_vitro_process__source_organism_iri,in_vitro_process__species,in_vitro_process__species_iri,antigen_processing__comments,assay__location_of_assay_data_in_reference,assay__method,assay__response_measured,assay__units,assay__iri,assay__qualitative_measurement,assay__measurement_inequality,assay__quantitative_measurement,assay__number_of_subjects_tested,assay__number_of_subjects_responded,assay__response_frequency_,assay__pdb_id,assay__comments,antigen_presenting_cell__source_tissue,antigen_presenting_cell__source_tissue_iri,antigen_presenting_cell__name,antigen_presenting_cell__iri,antigen_presenting_cell__culture_condition,mhc_restriction__name,mhc_restriction__iri,mhc_restriction__evidence_code,mhc_restriction__evidence_iri,mhc_restriction__class
0,1318603,https://cedar.iedb.org/assay/1318603,https://cedar.iedb.org/reference/1002036,Literature,10689116,None,F Borrás-Cuesta; J Golvano; M García-Granero; ...,Hum Immunol,2000,Specific and general HLA-DR binding motifs: co...,https://cedar.iedb.org/epitope/72258,Linear peptide,WAQPGYPWPLYGNE,c76-89,None,None,76,89,None,None,Core protein,http://www.ncbi.nlm.nih.gov/protein/Q5I9E3,Genome polyprotein,http://www.uniprot.org/uniprot/P27958,Hepatitis C virus subtype 1b,http://purl.obolibrary.org/obo/NCBITaxon_31647,Hepacivirus hominis,http://purl.obolibrary.org/obo/NCBITaxon_3052230,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,None,None,None,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Figure 1,cellular MHC/competitive/fluorescence,qualitative binding,None,http://purl.obolibrary.org/obo/OBI_0001594,Negative,None,NaN,None,None,None,None,The relative inhibitory capacity (RI) of pepti...,None,None,None,None,None,HLA-DRB1*01:01,http://purl.obolibrary.org/obo/MRO_0001279,None,None,II
1,1435717,https://cedar.iedb.org/assay/1435717,https://cedar.iedb.org/reference/1006117,Literature,8763995,None,C Pique; F Connan; J P Levilain; J Choppin; M ...,

Now we will format the `iedb_assay_ids` to include them in the search parameters.

In [5]:
# Extract and format assay IDs from the epitope_search result
assay_ids_raw = epitope_df['cedar_assay_ids'].iloc[0]

# Remove braces and create comma-separated string
cedar_assay_ids = assay_ids_raw.replace('{', '').replace('}', '')

print(cedar_assay_ids)

3000175,7689417,21249142,21249144,21249146,21527266,21527267,21527268,22330197,22396026,22704086,22704087,25149975,25149977,25149986,25149993,25149998,25149999,25150001,25150003,25150005,25150009,25150018,25150027,25150031,25150042,25150044


In this example, we are looking for assays performed in human subjects (cancer patients). CEDAR is connected to external resources using IRIs (Internationalized Resource Identifiers) that link to databases such as the [NCBI Taxonomy](https://www.ncbi.nlm.nih.gov/taxonomy), allowing users to filter the data using stable identifiers. The human organism NCBI Taxonomy ID is [9606](https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?mode=Info&id=9606&lvl=3&lin=f&keep=1&srchmode=1&unlock).

The field `host__iri` contains the IRI of the organism whose immune response is being measured. To construct the proper IRI for API queries, we must retrieve the IRI prefix from CEDAR's `curie_map` endpoint and join it with the Taxonomy ID.

In [6]:
endpoint = 'curie_map'

search_params = {
    # Query filter for NCBI Taxonomy prefix
    'curie_prefix': 'eq.NCBITaxon',

    # Select only the IRI replacement pattern
    'select': 'iri_replace',
}

# Retrieve IRI prefix
iri_df = API_query(endpoint, search_params)
ncbi_iri_prefix = iri_df.iloc[0, 0]

print(f"NCBI Taxonomy IRI prefix: {ncbi_iri_prefix}")

# Construct complete human organism IRI
human_iri = ncbi_iri_prefix + '9606'

print(f"Human IRI: {human_iri}")

Fetching offset: 0
Fetching offset: 10000
NCBI Taxonomy IRI prefix: http://purl.obolibrary.org/obo/NCBITaxon_
Human IRI: http://purl.obolibrary.org/obo/NCBITaxon_9606


Now we build the query, filtering by the assay IDs and the human host organism IRI.

In [7]:
endpoint='mhc_export'

search_params = {
    # Query filter
    'assay_id': 'in.(' + cedar_assay_ids + ')',
    'host__iri':'eq.' + human_iri,

    # Pagination
    'order': 'assay_id',
    'offset': 0,
}

# Execute query
mhc_assays_df = API_query(endpoint, search_params)

print(f"Retrieved {len(mhc_assays_df)} MHC elution assays")
mhc_assays_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 8 MHC elution assays


,assay_id,assay_id__cedar_iri,reference__cedar_iri,reference__type,reference__pmid,reference__submission_id,reference__authors,reference__journal,reference__date,reference__title,epitope__epitope_iri,epitope__object_type,epitope__name,epitope__reference_name,epitope__modified_residues,epitope__modifications,epitope__starting_position,epitope__ending_position,epitope__iri,epitope__synonyms,epitope__source_molecule,epitope__source_molecule_iri,epitope__molecule_parent,epitope__molecule_parent_iri,epitope__source_organism,epitope__source_organism_iri,epitope__species,epitope__species_iri,epitope__mutation,epitope__comments,related_object__epitope_relation,related_object__object_type,related_object__name,related_object__starting_position,related_object__ending_position,related_object__iri,related_object__synonyms,related_object__source_molecule,related_object__source_molecule_iri,related_object__molecule_parent,related_object__molecule_parent_iri,related_object__source_organism,related_object__source_organism_iri,related_object__species,related_object__species_iri,host__name,host__iri,host__geolocation,host__geolocation_iri,host__sex,...,in_vivo_antigen__source_molecule_iri,in_vivo_antigen__molecule_parent,in_vivo_antigen__molecule_parent_iri,in_vivo_antigen__source_organism,in_vivo_antigen__source_organism_iri,in_vivo_antigen__species,in_vivo_antigen__species_iri,in_vivo_antigen__adjuvants,in_vivo_antigen__route,in_vivo_antigen__dose_schedule,in_vitro_process__process_type,in_vitro_process__epitope_relation,in_vitro_process__object_type,in_vitro_process__name,in_vitro_process__reference_name,in_vitro_process__starting_position,in_vitro_process__ending_position,in_vitro_process__iri,in_vitro_process__source_molecule,in_vitro_process__source_molecule_iri,in_vitro_process__molecule_parent,in_vitro_process__molecule_parent_iri,in_vitro_process__source_organism,in_vitro_process__source_organism_iri,in_vitro_process__species,in_vitro_process__species_iri,antigen_processing__comments,assay__location_of_assay_data_in_reference,assay__method,assay__response_measured,assay__units,assay__iri,assay__qualitative_measurement,assay__measurement_inequality,assay__quantitative_measurement,assay__number_of_subjects_tested,assay__number_of_subjects_responded,assay__response_frequency_,assay__pdb_id,assay__comments,antigen_presenting_cell__source_tissue,antigen_presenting_cell__source_tissue_iri,antigen_presenting_cell__name,antigen_presenting_cell__iri,antigen_presenting_cell__culture_condition,mhc_restriction__name,mhc_restriction__iri,mhc_restriction__evidence_code,mhc_restriction__evidence_iri,mhc_restriction__class
0,3000175,https://cedar.iedb.org/assay/3000175,https://cedar.iedb.org/reference/1031072,Submission,NaN,1000715.0,"Zeynep Koşaloğlu-Yalçın, Manasa Lanka (1), Ang...",NaN,2016,Predicting T cell recognition of MHC class I r...,https://cedar.iedb.org/epitope/62627,Linear peptide,SYLDSGIHF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"S30F,S37F",NaN,in-frame neo-epitope,Fragment of a Natural Sequence Molecule,SYLDSGIHS,29,37,NaN,"Beta-catenin, CTNNB1, CTNNB, OK/SW-cl.35, PRO2286",Catenin beta-1,https://www.uniprot.org/uniprot/P35222.1,Catenin beta-1,http://www.uniprot.org/uniprot/P35222,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens,http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,purified MHC/competitive/radioactivity,dissociation constant KD (~IC50),nM,http://purl.obolibrary.org/obo/OBI_0001564,Positive-High,=,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*24:02,http://purl.obolibrary.org/obo/MRO_0001032,NaN,NaN,I
1,21249142,https://cedar.iedb.org/assay/21249142,https://cedar.iedb.org/reference/315132,Literature,8642260.0,NaN,P F Robbins; M El-Gamil; Y F Li; Y Kawakami; D...,J Exp Med,1996,A mutat

### Step 3: Map associated T cell assays

Similarly, with the assay IDs of interest, we map all the information of the associated T cell assays in humans by querying the ``tcell_export`` endpoint.

In [8]:
endpoint='tcell_export'

search_params = {
    # Query filter
    'assay_id': 'in.(' + cedar_assay_ids + ')',
    'host__iri':'eq.' + human_iri,

    # Pagination
    'order': 'assay_id',
    'offset': 0,
}

# Execute final query
tcell_assays_df = API_query(endpoint, search_params)

print(f"Retrieved {len(tcell_assays_df)} T cell assays")
tcell_assays_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 16 T cell assays


,assay_id,assay_id__cedar_iri,reference__cedar_iri,reference__type,reference__pmid,reference__submission_id,reference__authors,reference__journal,reference__date,reference__title,epitope__cedar_iri,epitope__object_type,epitope__name,epitope__reference_name,epitope__modified_residues,epitope__modifications,epitope__starting_position,epitope__ending_position,epitope__iri,epitope__synonyms,epitope__source_molecule,epitope__source_molecule_iri,epitope__molecule_parent,epitope__molecule_parent_iri,epitope__source_organism,epitope__source_organism_iri,epitope__species,epitope__species_iri,epitope__mutation,epitope__epitope_comments,related_object__epitope_relation,related_object__object_type,related_object__name,related_object__starting_position,related_object__ending_position,related_object__iri,related_object__synonyms,related_object__source_molecule,related_object__source_molecule_iri,related_object__molecule_parent,related_object__molecule_parent_iri,related_object__source_organism,related_object__source_organism_iri,related_object__species,related_object__species_iri,host__name,host__iri,host__geolocation,host__geolocation_iri,host__sex,...,in_vitro_immunogen__source_organism_iri,in_vitro_immunogen__species,in_vitro_immunogen__species_iri,adoptive_transfer__flag,adoptive_transfer__comments,immunization__comments,assay__location_of_assay_data_in_reference,assay__method,assay__response_measured,assay__units,assay__iri,assay__qualitative_measurement,assay__measurement_inequality,assay__quantitative_measurement,assay__number_of_subjects_tested,assay__number_of_subjects_positive,assay__response_frequency_,assay__comments,effector_cell__source_tissue,effector_cell__source_tissue_iri,effector_cell__name,effector_cell__iri,effector_cell__culture_condition,effector_cell__tcr_name,complex__pdb_id,antigen_presenting_cell__source_tissue,antigen_presenting_cell__source_tissue_iri,antigen_presenting_cell__name,antigen_presenting_cell__iri,antigen_presenting_cell__culture_condition,mhc_restriction__name,mhc_restriction__iri,mhc_restriction__evidence_code,mhc_restriction__evidence_iri,mhc_restriction__class,assay_antigen__epitope_relation,assay_antigen__object_type,assay_antigen__name,assay_antigen__reference_name,assay_antigen__starting_position,assay_antigen__ending_position,assay_antigen__iri,assay_antigen__source_molecule,assay_antigen__source_molecule_iri,assay_antigen__molecule_parent,assay_antigen__molecule_parent_iri,assay_antigen__source_organism,assay_antigen__source_organism_iri,assay_antigen__species,assay_antigen__species_iri
0,7689417,https://cedar.iedb.org/assay/7689417,https://cedar.iedb.org/reference/1036848,Literature,32314731,NaN,Kenji Murata; Munehide Nakatsugawa; Muhammed A...,Elife,2020,Landscape mapping of shared antigenic epitopes...,https://cedar.iedb.org/epitope/62627,Linear peptide,SYLDSGIHF,A*24:02 Peptide 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"S30F,S37F",NaN,in-frame neo-epitope,Fragment of a Natural Sequence Molecule,SYLDSGIHS,29,37,NaN,"Beta-catenin, CTNNB1, CTNNB, OK/SW-cl.35, PRO2286",Catenin beta-1,https://www.uniprot.org/uniprot/P35222.1,Catenin beta-1,http://www.uniprot.org/uniprot/P35222,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens,http://purl.obolibrary.org/obo/NCBITaxon_9606,Homo sapiens (human),http://purl.obolibrary.org/obo/NCBITaxon_9606,NaN,NaN,NaN,...,NaN,NaN,NaN,N,NaN,TILs isolated from metastatic melanoma patient...,"Figure 1, Table 1",multimer/tetramer,qualitative binding,NaN,http://purl.obolibrary.org/obo/OBI_1110179,Negative,NaN,NaN,3.0,0.0,0.0,NaN,skin,http://purl.obolibrary.org/obo/UBERON_0002097,T cell CD8+,http://purl.obolibrary.org/obo/CL_0000625,Direct Ex Vivo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HLA-A*24:02,http://purl.obolibrary.org/obo/MRO_0001032,Cited reference,http://purl.obolibrary.org/obo/ECO_0000304,I,Epitope,Linear peptide,SYLDSGIHF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21249144,https://cedar.iedb.org/assay/21249144,https:

### Results

We have obtained two final DataFrames containing the MHC elution and T cell assays of the neoepitope *SYLDSGIHF*. This programmatic approach demonstrates the CEDAR API's capability for automated data retrieval.

## References

Koşaloğlu-Yalçın, Z. et al. (2025) *Using the Cancer Epitope Database and Analysis Resource (CEDAR)*. Alexander Krasnitz and Pascal Belleau (Eds.), *Cancer Bioinformatics, Methods in Molecular Biology*, vol. 2932. [https://doi.org/10.1007/978-1-0716-4566-6_3 ](https://doi.org/10.1007/978-1-0716-4566-6_3)

Koşaloğlu-Yalçın, Z. et al. (2023) *The cancer epitope database and analysis resource (CEDAR).* Nucleic acids research 51, no. D1 (2023): D845-D852. [https://doi.org/10.1093/nar/gkac902 ](https://doi.org/10.1093/nar/gkac902)